# Lin 2019 HPF-residual CNN — using the existing Steganalysis pipeline
Uses your own `StegoDataset` / `StegoTrainer` (same `GroupShuffleSplit` cover-pairing and the same fixed 10-seed run pattern `[42,101,202,...]` already used for CNN/BiLSTM/SVM). The only new pieces are `get_raw_waveform()` (time-domain input, not spectrogram) and `build_lin2019_model()` (SRM HPF residual + Conv1D blocks) added to `features.py` / `models.py`.

## 0. Setup

In [ ]:
!pip install -q tensorflow librosa psutil pytz seaborn scikit-learn
!git clone https://github.com/Linyuzhen/audio_steganalysis.git lin2019_repo
!cp lin2019_repo/SRM_K.npy ./SRM_K.npy

Cloning into 'lin2019_repo'...
remote: Enumerating objects: 62, done.
remote: Total 62 (delta 0), reused 0 (delta 0), pack-reused 62 (from 1)
Receiving objects: 100% (62/62), 17.03 KiB | 17.03 MiB/s, done.
Resolving deltas: 100% (29/29), done.


In [ ]:
# from google.colab import files
# import os
# import re

# def clean_file(filename):
#     return re.sub(r'\(\d+\)(?=\.[^.]+$)', '', filename)


# print('Upload the patched pipeline files: dataset.py, features.py, models.py, trainer.py')

# uploaded = files.upload()

# os.makedirs('Steganalysis', exist_ok=True)

# for old_name in uploaded.keys():
#     new_name = clean_file(old_name)

#     src = old_name
#     dst = os.path.join('Steganalysis', new_name)

#     os.replace(src, dst)
#     print(f"{old_name} -> {dst}")

# open('Steganalysis/__init__.py', 'w').close()

In [ ]:
!rm -r content

## 1. Mount Drive (where your 1500 pre-generated cover/stego files live)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!unzip -q /content/drive/MyDrive/HK1-20252026/NEW-RUN-STEGO/Steganalysis.zip -d ./
!unzip -q /content/drive/MyDrive/HK1-20252026/NEW-RUN-STEGO/cache.zip -d ./

## 2. Point to your existing 1500 M5 cover/stego pairs (already generated -- no re-encoding needed)

In [ ]:
# EDIT these two paths to wherever your 1500 pre-generated cover/stego files live
# (e.g. on Drive: '/content/drive/MyDrive/your_folder/cover_dir')
COVER_DIR = "/content/drive/MyDrive/HK1-20252026/Steganography/Data/musdb18-hq/train"
STEGO_DIR =  "/content/drive/MyDrive/HK1-20252026/Steganography/Ablation/5_Random_Adaptive_ContentSalt"

import os
assert os.path.isdir(COVER_DIR), f'Not found: {COVER_DIR}'
assert os.path.isdir(STEGO_DIR), f'Not found: {STEGO_DIR}'

cover_files = sorted(f for f in os.listdir(COVER_DIR) if f.lower().endswith(('.wav', '.flac', '.mp3')))
stego_files = sorted(f for f in os.listdir(STEGO_DIR) if f.lower().endswith(('.wav', '.flac', '.mp3')))

print(f'Cover files: {len(cover_files)}')
print(f'Stego files: {len(stego_files)}')

# StegoDataset pairs cover[i] <-> stego[i] by SORTED FILENAME ORDER (see dataset.py _scan_files()).
# If filenames match 1-to-1 between the two folders, this check should pass:
mismatches = [(a, b) for a, b in zip(cover_files, stego_files) if a != b]
if mismatches:
    print(f'[WARNING] {len(mismatches)} filename mismatches between cover/stego at the same sorted position!')
    print('First few:', mismatches[:5])
    print('StegoDataset will still pair them by position, but double check this is intended.')
else:
    print('Filenames match 1-to-1 in sorted order -- safe to proceed.')

if len(cover_files) != len(stego_files):
    print(f'[NOTE] Count mismatch ({len(cover_files)} vs {len(stego_files)}) -- '
          f'StegoDataset.get_train_val_split() will use min(n_cover, n_stego) pairs automatically.')

Cover files: 750
Stego files: 750
Filenames match 1-to-1 in sorted order -- safe to proceed.


In [ ]:
# Copy to local disk before training -- reading 1500x2 files repeatedly straight off the
# mounted Drive (FUSE + network I/O) is slow and prone to crashes/timeouts during training.
import shutil, time
LOCAL_COVER_DIR = '/content/cover_dir'
LOCAL_STEGO_DIR = '/content/stego_dir'

# for src, dst in [(COVER_DIR, LOCAL_COVER_DIR), (STEGO_DIR, LOCAL_STEGO_DIR)]:
#     if not os.path.isdir(dst):
#         print(f'Copying {src} -> {dst} ...')
#         t0 = time.time()
#         shutil.copytree(src, dst)
#         print(f'  done in {time.time()-t0:.1f}s')
#     else:
#         print(f'{dst} already exists, skipping copy.')

COVER_DIR, STEGO_DIR = LOCAL_COVER_DIR, LOCAL_STEGO_DIR  # all subsequent steps use the local copy

## 3. Train using StegoTrainer -- exactly the same 10-run / GroupShuffleSplit pattern you already use for CNN/BiLSTM/SVM
`StegoDataset.get_train_val_split()` already does cover-disjoint `GroupShuffleSplit` 70/15/15 (test split fixed at `random_state=42`, train/val split varies by `random_state=seed`). `StegoTrainer.train(runs=10)` already loops over the fixed seed list `[42,101,202,303,404,505,606,707,808,909]` -- identical to your other algorithms, no new splitting logic introduced.

In [ ]:
import sys
import time
sys.path.insert(0, '.')
from content.Steganalysis.trainer import StegoTrainer

# ----------------------------------------------------------------
# NOTE: batch_size reduced from the original Lin (2019) config of 64
# to 8, due to GPU memory constraints on this Colab runtime.
# ReduceLROnPlateau (factor=0.5, patience=3, min_lr=1e-7) is already
# enabled inside StegoTrainer.train() -> it will automatically halve
# the learning rate if val_loss plateaus for 3 consecutive epochs,
# partially compensating for the increased gradient noise from the
# smaller batch size. No manual LR adjustment is needed here.
# ----------------------------------------------------------------

BATCH_SIZE = 16
LEARNING_RATE = 0.0001
EPOCHS = 10
RUNS = 7

print("=" * 60)
print("Lin 2019 HPF-residual CNN — Steganalysis Experiment")
print("=" * 60)
print(f"Config: epochs={EPOCHS}, batch={BATCH_SIZE}, "
      f"lr={LEARNING_RATE}, runs={RUNS}")
print(f"Note  : batch_size reduced from original 64 -> {BATCH_SIZE} "
      f"(Colab GPU memory limit). ReduceLROnPlateau active "
      f"(factor=0.5, patience=3, min_lr=1e-7).")
print(f"Cover : {COVER_DIR}")
print(f"Stego : {STEGO_DIR}")
print("=" * 60)

t_total_start = time.time()

trainer = StegoTrainer(
    cover_dir=COVER_DIR,
    stego_dir=STEGO_DIR,
    algo='lin2019',
    cache_dir='/content/content/cache',
    log_dir='/content/drive/MyDrive/HK1-20252026/NEW-RUN-STEGO/0-M5/logs',
    srm_kernel_path='/content/lin2019_repo/SRM_K.npy'
)

summary = trainer.train(
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LEARNING_RATE,
    runs=RUNS
)

t_elapsed = time.time() - t_total_start
hours = int(t_elapsed // 3600)
mins = int((t_elapsed % 3600) // 60)
secs = int(t_elapsed % 60)

print("\n" + "=" * 60)
print("EXPERIMENT COMPLETE")
print("=" * 60)
print(f"Total wall time : {hours:02d}h {mins:02d}m {secs:02d}s")
print(f"Accuracy : {summary['mean_acc']*100:.2f}% ± {summary['std_acc']*100:.2f}%")
print(f"AUC      : {summary['mean_auc']:.4f} ± {summary['std_auc']:.4f}")
print("=" * 60)
print("\n>>> Copy these into Table 6 and Table 11:")
print(f"  Lin 2019 | {summary['mean_acc']*100:.2f}±{summary['std_acc']*100:.2f}% | "
      f"{summary['mean_auc']:.4f}±{summary['std_auc']:.4f}")
print(f"\n>>> Reproducibility note for the manuscript:")
print(f"  batch_size={BATCH_SIZE} (original repo: 64, reduced for Colab GPU memory); "
      f"ReduceLROnPlateau enabled to offset gradient-noise increase.")

Lin 2019 HPF-residual CNN — Steganalysis Experiment
Config: epochs=10, batch=16, lr=0.0001, runs=7
Note  : batch_size reduced from original 64 -> 16 (Colab GPU memory limit). ReduceLROnPlateau active (factor=0.5, patience=3, min_lr=1e-7).
Cover : /content/cover_dir
Stego : /content/stego_dir

[Trainer] Starting experiment: Algo=LIN2019 | LR=0.0001
[Log] Base Directory: /content/drive/MyDrive/HK1-20252026/NEW-RUN-STEGO/0-M5/logs/LIN2019_D3_F32_LR0.0001_20260629-152556
[Data] Loading dataset for model framework (LIN2019)...
[Cache] Found cache file: /content/content/cache/features_lin2019.npz

--- Run 1/7 (Random Seed: 808) ---
[Fast Load] Loading from cache...
 -> Loaded: X=(1500, 220500), y=(1500,)
[Auto-Fix] Reshaping 2D -> 3D channel for Conv1D: (1500, 220500) -> (1500, 220500, 1)
[build_lin2019_model] Loaded SRM kernel raw shape: (4, 5), dtype: int32
[build_lin2019_model] HPF Conv1D kernel tensor shape: (5, 1, 4) (kernel_size=5, in_channels=1, filters=4)
Epoch 1/10
65/65 ━━━━━━━━━━━

KeyboardInterrupt: 

## 4. Zip + download all logs (per-run CSVs, charts, summary) for your records

In [ ]:
!zip -rq lin2019_logs.zip Steganalysis
from google.colab import files
files.download('lin2019_logs.zip')

In [ ]:
!zip -rq lin2019_data.zip /content/cache/features_lin2019_parts
from google.colab import files
files.download('lin2019_data.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:

import os
os.makedirs('/content/drive/MyDrive/HK1-20252026/NEW-RUN-STEGO', exist_ok=True)

!zip -rq /content/drive/MyDrive/HK1-20252026/NEW-RUN-STEGO/cache.zip /content/cache



In [ ]:

import os
os.makedirs('/content/drive/MyDrive/HK1-20252026/NEW-RUN-STEGO', exist_ok=True)

!zip -rq /content/drive/MyDrive/HK1-20252026/NEW-RUN-STEGO/Steganalysis.zip /content/Steganalysis

